# ML-08 — Capstone Modeling Lane

**Author:** Mehak Zahra  
**Lane:** Refresh / Content Opportunity Scoring  
**Primary metric:** Precision@50

The target is the supplied observed-decline proxy. Probabilities are used only to rank pages for human review; this is not a causal refresh-impact model or a claim about Google's algorithm.

## 1. Method choice and why

This is a yes/no proxy supporting a ranked decision. I start with **Logistic Regression** because its probability provides a ranking and its behavior is easier to inspect. I also test a constrained **Random Forest** to learn whether non-linear interactions earn their complexity. Both must beat the Week-4 frozen rule on the same held-out rows and Precision@50; complexity receives no credit by itself.

The feature set uses pre-existing page/search/content measurements. Direct label fields and the last/previous-30-day components that create the label are excluded. Missing numeric values receive median imputation plus missingness indicators because missingness is patterned; categories use most-frequent imputation and one-hot encoding.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_score, recall_score, f1_score)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]]
            if (p / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists())
df = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
y = df['trend_direction'].eq('down').astype(int)
SEED = 42

NUMERIC = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
           'impressions_90d', 'clicks_90d', 'sessions_90d', 'days_with_impressions',
           'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr',
           'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
CATEGORICAL = ['competition_level', 'content_type', 'main_intent', 'age_tier',
               'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
FEATURES = NUMERIC + CATEGORICAL
FORBIDDEN = {'trend_direction', 'trend_pct', 'is_declining_label',
             'impressions_last_30d', 'impressions_prev_30d',
             'clicks_last_30d', 'clicks_prev_30d',
             'sessions_last_30d', 'sessions_prev_30d'}
assert set(FEATURES).isdisjoint(FORBIDDEN)
print(f'Loaded {len(df):,} rows; {len(FEATURES)} safe source features; scikit-learn {sklearn.__version__}.')

Loaded 30,000 rows; 25 safe source features; scikit-learn 1.8.0.


## 2. Split design

`GroupShuffleSplit` holds out 20% of the 32 pseudonymized clients. A page from a test client can never appear in training, so the evaluation asks whether the ranking transfers to entirely unseen client groups. Seed 42 fixes the split. This is stronger than a random page split for client transfer, although one grouped holdout is not a final production estimate.

In [2]:
X = df[FEATURES]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(X, y, groups=df['client_id']))
train_clients = set(df.iloc[train_idx]['client_id'])
test_clients = set(df.iloc[test_idx]['client_id'])
assert train_clients.isdisjoint(test_clients)
print(f'Train: {len(train_idx):,} rows / {len(train_clients)} clients')
print(f'Test:  {len(test_idx):,} rows / {len(test_clients)} clients')
print('Client overlap: 0')
print(f'Test base rate: {y.iloc[test_idx].mean():.3f}')

Train: 23,837 rows / 25 clients
Test:  6,163 rows / 7 clients
Client overlap: 0
Test base rate: 0.511


## 3. Train and compare with the frozen Week-4 baseline

The Week-4 score is recomputed unchanged: visibility percentile × position opportunity × CTR gap × eligibility. Baseline and learned models are evaluated on exactly the same 6,163 held-out rows. Precision@10 and Precision@50 measure the queue head; ROC AUC and average precision describe ranking across thresholds.

In [3]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

# Frozen ML-07 baseline: no changed threshold or weight.
visibility = np.log1p(df['impressions_90d']).rank(method='average', pct=True)
position_opportunity = (((20 - df['avg_position']) / 19).clip(0, 1)
                        * df['avg_position'].between(0.000001, 20).astype(int))
ctr_gap = ((0.50 - df['ctr']) / 0.50).clip(0, 1)
baseline_score = visibility * position_opportunity * ctr_gap * (df['impressions_90d'] >= 100)

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                          ('scale', StandardScaler())]), NUMERIC),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAL),
])
models = {
    'Logistic regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED),
    'Random forest': RandomForestClassifier(n_estimators=250, max_depth=12, min_samples_leaf=10,
                                             class_weight='balanced', random_state=SEED, n_jobs=-1),
}
fitted = {}
scores = {'Frozen rule baseline': baseline_score.iloc[test_idx].to_numpy()}
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
    fitted[name] = pipe
    scores[name] = pipe.predict_proba(X.iloc[test_idx])[:, 1]

y_test = y.iloc[test_idx].to_numpy()
rows = []
for name, score in scores.items():
    predicted = (np.asarray(score) >= 0.5).astype(int)
    rows.append({'method': name, 'base_rate': y_test.mean(),
                 'precision_at_10': precision_at_k(y_test, score, 10),
                 'precision_at_50': precision_at_k(y_test, score, 50),
                 'roc_auc': roc_auc_score(y_test, score),
                 'average_precision': average_precision_score(y_test, score),
                 'threshold_precision': precision_score(y_test, predicted, zero_division=0),
                 'recall': recall_score(y_test, predicted, zero_division=0),
                 'f1': f1_score(y_test, predicted, zero_division=0)})
comparison = pd.DataFrame(rows).set_index('method').round(3)
print(comparison.to_string())

                      base_rate  precision_at_10  precision_at_50  roc_auc  average_precision  threshold_precision  recall     f1
method                                                                                                                           
Frozen rule baseline      0.511              0.7             0.78    0.562              0.561                0.765   0.033  0.063
Logistic regression       0.511              0.8             0.74    0.580              0.577                0.563   0.633  0.596
Random forest             0.511              0.6             0.60    0.612              0.596                0.593   0.611  0.602


### Model-vs-baseline conclusion

The frozen baseline remains the operational winner at Precision@50: **0.78**, versus **0.74** for Logistic Regression and **0.60** for Random Forest. Logistic Regression improves Precision@10 from **0.70 to 0.80**, but that smaller cutoff was not the primary metric. Random Forest has the best ROC AUC (0.612) yet performs poorly at the queue head, showing why an overall metric cannot replace the decision metric.

I retain Logistic Regression as the learned comparison because it is simpler and beats the forest at both operational cutoffs. I do **not** claim that the model beat the baseline.

## 4. Interpretation and error analysis

Permutation importance measures the drop in held-out ROC AUC after shuffling one original feature. It is predictive, not causal. The top features are plausible activity/visibility signals rather than direct copies of the outcome.

In [4]:
selected = fitted['Logistic regression']
selected_score = scores['Logistic regression']
perm = permutation_importance(selected, X.iloc[test_idx], y.iloc[test_idx],
                              scoring='roc_auc', n_repeats=5,
                              random_state=SEED, n_jobs=-1)
importance = (pd.DataFrame({'feature': FEATURES, 'auc_drop': perm.importances_mean})
              .sort_values('auc_drop', ascending=False).head(8))
importance['auc_drop'] = importance['auc_drop'].round(4)
print(importance.to_string(index=False))

              feature  auc_drop
days_with_impressions    0.0476
   days_with_sessions    0.0419
     content_age_days    0.0260
         avg_position    0.0173
        position_tier    0.0056
          scroll_rate    0.0042
           word_count    0.0039
    competition_level    0.0030


The top three permutation signals are active impression days, active session days, and content age. Active-day counts plausibly distinguish sustained visibility and engagement from irregular exposure; age may capture lifecycle differences. Their modest AUC drops reinforce that no single feature explains the proxy.

Below are three high-confidence false positives and three high-confidence false negatives. These are pseudonymous pages only.

In [5]:
selected_pred = (selected_score >= 0.5).astype(int)
errors = df.iloc[test_idx][['content_id', 'client_id', 'impressions_90d',
                            'days_with_impressions', 'avg_position', 'ctr',
                            'content_age_days']].copy()
errors['actual'] = y_test
errors['predicted'] = selected_pred
errors['probability'] = selected_score
fp = errors[(errors.actual == 0) & (errors.predicted == 1)].sort_values(
    'probability', ascending=False).head(3)
fn = errors[(errors.actual == 1) & (errors.predicted == 0)].sort_values(
    'probability').head(3)
error_cases = pd.concat([fp.assign(error_type='false_positive'),
                         fn.assign(error_type='false_negative')])
error_cases['why_hard'] = np.where(
    error_cases.error_type.eq('false_positive'),
    'Strong historical activity can resemble risk even though the proxy did not decline.',
    'Low/irregular history gives the model little evidence despite the observed decline.')
error_view = error_cases[['error_type', 'content_id', 'probability',
                          'impressions_90d', 'days_with_impressions',
                          'avg_position', 'ctr', 'why_hard']].copy()
error_view['probability'] = error_view['probability'].round(3)
print(error_view.to_string(index=False))

    error_type           content_id  probability  impressions_90d  days_with_impressions  avg_position  ctr                                                                            why_hard
false_positive content_374e795aab68        0.920              235                     64          31.0 0.85 Strong historical activity can resemble risk even though the proxy did not decline.
false_positive content_7be5f150dc65        0.900              290                     53           5.9 0.00 Strong historical activity can resemble risk even though the proxy did not decline.
false_positive content_41baf0722ad9        0.884             3115                     88          12.8 0.00 Strong historical activity can resemble risk even though the proxy did not decline.
false_negative content_e18144cbd19d        0.065                3                      3           2.0 0.00 Low/irregular history gives the model little evidence despite the observed decline.
false_negative content_917fc1b11fe1     

False positives show that strong historical visibility can look risky without an observed decline; query intent or client-specific behavior may be missing. False negatives show that sparse/irregular pages can decline without the stable history the model relies on. The errors and grouped transfer gap support human review and argue against automatic content changes.

In [6]:
metrics = {
    'assignment': 'ML-08', 'author': 'Mehak Zahra',
    'lane': 'Refresh / Content Opportunity Scoring', 'seed': SEED,
    'split': 'GroupShuffleSplit by client_id; test_size=0.20',
    'train_rows': len(train_idx), 'test_rows': len(test_idx),
    'train_clients': len(train_clients), 'test_clients': len(test_clients),
    'selected_learned_model': 'Logistic regression',
    'operational_winner_at_50': 'Frozen rule baseline',
    'comparison': comparison.reset_index().to_dict(orient='records'),
    'top_permutation_features': importance.to_dict(orient='records'),
    'sklearn_version': sklearn.__version__,
}
output_dir = ROOT / 'work' / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'model_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')
assert set(FEATURES).isdisjoint(FORBIDDEN)
assert train_clients.isdisjoint(test_clients)
print('Checks passed; wrote work/outputs/model_metrics.json')

Checks passed; wrote work/outputs/model_metrics.json


## 5. Self-check

- [x] Method choice matches a probability-ranked binary proxy.
- [x] Client-group holdout has zero client overlap and fixed seed 42.
- [x] Baseline and models use the same test rows and metrics.
- [x] The final comparison table includes base rate, Precision@10/50, ROC AUC, average precision, precision, recall, and F1.
- [x] Model complexity is not rewarded: the baseline remains the Precision@50 winner.
- [x] Permutation importance and six concrete errors are interpreted.
- [x] Direct label and component fields are excluded from features.
- [x] All outputs are executed and visible; the metrics receipt is committed.